# AI Researcher Pipeline

## Task definition
Change this if you want to generate any new ideas.  
REFERENCES: List of paper titles that are relevant to the idea that you would want to generate. >5 required.  
MAX_DATE: The date after which you would like to generate the idea. This is usually the date of the most recent reference paper. 
It's only used for github search right now. 

In [ ]:
REFERENCES = [
    "Lightgcn: Simplifying and powering graph convolution network for recommendation",
    "Neural collaborative filtering",
    "Disentangled contrastive learning on graphs",
    "Improving Graph Collaborative Filtering with Neighborhood-enriched Contrastive Learning",
    "Curriculum Disentangled Recommendation with Noisy Multi-feedback",
    "Disentangled heterogeneous graph attention network for recommendation",
    "Learning intents behind interactions with knowledge graph for recommendation",
    "LightGCL: Simple Yet Effective Graph Contrastive Learning for Recommendation",
    "Self-supervised graph learning for recommendation",
]
assert len(REFERENCES) >= 5, "Please provide at least 5 reference paper titles."

import datetime
MAX_DATE = datetime.date(2023, 10, 1)
TASK_DESCRIPTION = "Generate an idea for a novel recommendation model."
MODEL_NAME = "gpt-5-nano"
# AI Researcher used GPT 4o. gpt-5-nano is better and cheaper. 
# Best and most expensive is gpt-5

# Alternatively, load this info directly from their provided files: 
# INPUT_DATA_PATH = os.path.join(Config.INPUT_DATA_PATH, 'dccf.json')
# task_metadata = load_input_data(INPUT_DATA_PATH)
# max_date = datetime.datetime.strptime(task_metadata['date_limit'], '%Y-%m-%d').date()
# references = get_references(task_metadata)

## Searching GitHub Repositories
First they look in github which repos are similar to the references provided

In [ ]:
from src.ai_researcher import Config
from src.ai_researcher.github_search import search_github_batched

github_result = search_github_batched(REFERENCES, max_date=MAX_DATE)

## Choosing reference codebases
Basicially, the user prompt asks to determine the 5 most promising codebases. 
The system prompt outlines how to use the various tools. 

In [ ]:
from src.ai_researcher.prompts.reference_codebases_prompt import get_reference_codebases_user_prompt, reference_codebases_system_prompt
from src.horizon_bench.openai_api import chat_with_agentic_tool_use
from src.ai_researcher.classes.Tools import ExecuteCommandTool, ReadFileTool, CodeStructureTool
from pydantic import BaseModel

user_prompt = get_reference_codebases_user_prompt(REFERENCES, github_result)
system_prompt = reference_codebases_system_prompt

class CodeBasesResult(BaseModel):
    codebases: list[str]
    paper_titles: list[str]

chosen_codebases = chat_with_agentic_tool_use(
    user_prompt=user_prompt,
    tools=[ExecuteCommandTool(), ReadFileTool(), CodeStructureTool()],
    system_prompt=system_prompt,
    response_format=CodeBasesResult,
    model=MODEL_NAME
)

## Downloading arxiv papers
They assume each reference paper is on arxiv. If a paper is not found, it is skipped.

In [ ]:
from src.horizon_bench.arxiv_search import search_download_arxiv, string_format_arxiv_path_title
arxiv_search_results = search_download_arxiv(chosen_codebases.paper_titles)
formatted_arxiv_search_results = string_format_arxiv_path_title(arxiv_search_results)

## Generate an idea
Multi turn chat with tool use to read the downloaded files. 

In [ ]:
from pydantic import BaseModel
from src.ai_researcher.classes.Tools import ReadFileTool
from src.horizon_bench.openai_api import chat_with_agentic_tool_use
from src.ai_researcher.prompts.idea_prompt import idea_user_prompt, idea_system_prompt

class IdeaResult(BaseModel):
    challenges: str
    existing_methods: str
    motivation: str
    proposed_method: str
    technical_details: str
    expected_outcomes: str


response = chat_with_agentic_tool_use(
    model=MODEL_NAME,
    user_prompt=idea_user_prompt.format(task=TASK_DESCRIPTION,
                references=chosen_codebases.paper_titles,
                github_codebases=chosen_codebases.codebases,
                download_res=formatted_arxiv_search_results),
    tools=[ReadFileTool()],
    system_prompt=idea_system_prompt,
    response_format=IdeaResult,
    # reasoning={
    #     "effort": "high"  # Options: "low", "medium" (default), "high"
    # },
)


## Showing Idea generation results

In [ ]:
from src.horizon_bench.markdown import string_to_markdown

string_to_markdown('### Challenges:')
string_to_markdown(response.challenges)
string_to_markdown('### Existing Methods:')
string_to_markdown(response.existing_methods)
string_to_markdown('### Motivation:')
string_to_markdown(response.motivation)
string_to_markdown('### Proposed Method:')
string_to_markdown(response.proposed_method)
string_to_markdown('### Technical Details:')
string_to_markdown(response.technical_details)
string_to_markdown('### Expected Outcomes:')
string_to_markdown(response.expected_outcomes)